In [21]:
from pathlib import Path
import sys
import os
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
LABELS_PATH = DATA_DIR / "labels.csv"
MODEL_PATH = MODELS_DIR / "best_model.pkl"
METRICS_PATH = RESULTS_DIR / "metrics.csv"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
from src.classifier import load_labeled_data, train_and_save_model

labels_df = load_labeled_data(str(LABELS_PATH))
labels_df.head()

,candidate_id,page_stub,page_url,image_url,local_path,domain,file_name,width,height,format,...,file_size_bytes,area,aspect_ratio,is_downloaded_and_valid,label,split,is_tiny,is_suspicious_domain,has_ui_keyword,target
0,page_01_cand_000000,page_01,https://eda.rambler.ru/media/recepty/pyat-horo...,https://mc.yandex.ru/watch/105674087,c:\Users\Masha\ML_Projects\Photo-Classificatio...,mc.yandex.ru,105674087,1.0,1.0,GIF,...,43,1.0,1.0,True,non_content,train,True,True,False,0
1,page_01_cand_000001,page_01,https://eda.rambler.ru/media/recepty/pyat-horo...,https://mc.yandex.ru/watch/27509004,c:\Users\Masha\ML_Projects\Photo-Classificatio...,mc.yandex.ru,27509004,1.0,1.0,GIF,...,43,1.0,1.0,True,non_content,train,True,True,False,0
2,page_01_cand_000002,page_01,https://eda.rambler.ru/media/recepty/pyat-horo...,https://mc.yandex.ru/watch/26649402,c:\Users\Masha\ML_Projects\Photo-Classificatio...,mc.yandex.ru,26649402,1.0,1.0,GIF,...,43,1.0,1.0,True,non_content,train,True,True,False,0
3,page_01_cand_000003,page_01,https://eda.rambler.ru/media/recepty/pyat-horo...,https://counter.rambler.ru/top100.cnt?pid=2635991,c:\Users\Masha\ML_Projects\Photo-Classificatio...,counter.rambler.ru,top100.cnt,1.0,1.0,GIF,...,43,1.0,1.0,True,non_content,train,True,True,True,0
4,page_01_cand_000004,page_01,https://eda.rambler.ru/media/recepty/pyat-horo...,https://counter.rambler.ru/top100.cnt?pid=7728281,c:\Users\Masha\ML_Projects\Photo-Classificatio...,counter.rambler.ru,top100.cnt,1.0,1.0,GIF,...,43,1.0,1.0,True,non_content,train,True,True,True,0


In [22]:
labels_df['label'].value_counts(), labels_df['split'].value_counts()

(label
 content        306
 non_content    221
 Name: count, dtype: int64,
 split
 train    323
 test     102
 val      102
 Name: count, dtype: int64)

In [23]:
report = train_and_save_model(
    labels_csv_path=str(LABELS_PATH),
    model_path=str(MODEL_PATH),
    model_type='logreg',
)
report['threshold']

0.76

In [24]:
metrics_df = pd.DataFrame([
    {'split': 'train', **report['train_metrics']},
    {'split': 'val', **report['val_metrics']},
    {'split': 'test', **report['test_metrics']},
])
metrics_df[['split', 'precision', 'recall', 'f1', 'tp', 'fp', 'fn', 'tn']]

,split,precision,recall,f1,tp,fp,fn,tn
0,train,0.946237,0.455959,0.615385,88,5,105,125
1,val,0.863636,0.316667,0.463415,19,3,41,39
2,test,0.862069,0.471698,0.609756,25,4,28,45


In [25]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
metrics_df.to_csv(METRICS_PATH, index=False)
metrics_df

,split,precision,recall,f1,accuracy,tp,fp,fn,tn,n,mean_proba,median_proba,proba_p90
0,train,0.946237,0.455959,0.615385,0.659443,88,5,105,125,323,NaN,NaN,NaN
1,val,0.863636,0.316667,0.463415,0.568627,19,3,41,39,102,0.510476,0.479203,0.845656
2,test,0.862069,0.471698,0.609756,0.686275,25,4,28,45,102,0.552689,0.577540,0.840928
